# Pipeline 4: Intervention Effectiveness & Recommendation
## Lighthouse Sanctuary — ML Pipeline Notebook

---

### Pipeline Thinking (Ch. 1 / Foreword)
Social workers currently select interventions based on experience and intuition. This pipeline asks a structured question: *"Among residents with a similar profile to this one, which combination of services, session types, and frequencies produced the strongest improvements?"* It is not a fully supervised prediction problem — it is a **clustering and within-cluster effectiveness ranking** system, which is the right tool for a recommendation problem at this scale and data availability.

The story runs from a child-welfare decision-support problem, through longitudinal outcome computation, rule-based profile clustering, within-cluster service ranking, and a deployed recommendation artifact that writes per-resident recommendations to the social worker dashboard.

---

### Prediction vs. Explanation (Ch. 9–11)
**This is a HYBRID pipeline — clustering for explanation, ranking for decision support.**

The goal is not to predict a specific outcome value (that would be prediction) nor to explain population-level associations (that would be explanation). The goal is to **find what worked** for similar residents and surface those patterns as structured recommendations for social workers.

Key modelling choices:
- **Rule-based hierarchical clustering** (not K-Means) — at ~60 residents, rule-based grouping by case_category + risk_level + age_band produces deterministic, interpretable clusters that social workers can reason about
- **Within-cluster ranking** by observed composite outcome improvement — this is a data-driven ranking, not a causal claim
- Recommendations include **confidence tiers** (high/medium/low) based on cluster size, explicitly communicating uncertainty

**What this pipeline does NOT claim:** It does not claim that the recommended service *caused* the outcome improvement. It reflects historical co-occurrence for similar residents. Social workers should use this as structured decision support, not a prescription.


In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, json
from datetime import datetime

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
SNAPSHOT = pd.Timestamp('2026-01-15')
print("All imports OK")


All imports OK


## Phase 2 — Data Acquisition

In [2]:
# ── Load all CSVs ─────────────────────────────────────────────────────────────
residents  = pd.read_csv('./csvs/residents.csv',
                         parse_dates=['date_of_admission','date_closed','created_at'])
health     = pd.read_csv('./csvs/health_wellbeing_records.csv', parse_dates=['record_date'])
education  = pd.read_csv('./csvs/education_records.csv',        parse_dates=['record_date'])
incidents  = pd.read_csv('./csvs/incident_reports.csv',         parse_dates=['incident_date'])
recordings = pd.read_csv('./csvs/process_recordings.csv',       parse_dates=['session_date'])

print("Loaded tables:")
for name, df in [('residents',residents),('health',health),('education',education),
                 ('incidents',incidents),('recordings',recordings)]:
    print(f"  {name:12s}: {df.shape}")


Loaded tables:
  residents   : (60, 49)
  health      : (534, 14)
  education   : (534, 10)
  incidents   : (100, 12)
  recordings  : (2819, 15)


## Phase 3 — Exploratory Data Analysis (Ch. 6, 8)

In [3]:
# ── Profile distribution: how many residents per case_category × risk_level? ──
profile_dist = (residents
    .groupby(['case_category','current_risk_level'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False))
print("Resident profile distribution (case_category × risk_level):")
print(profile_dist.to_string())
print(f"\nTotal active + closed residents: {len(residents)}")


Resident profile distribution (case_category × risk_level):
   case_category current_risk_level  count
11   Surrendered                Low     14
2      Abandoned                Low      9
3      Abandoned             Medium      7
5      Foundling                Low      6
8      Neglected                Low      5
12   Surrendered             Medium      5
6      Foundling             Medium      4
9      Neglected             Medium      4
10   Surrendered               High      2
0      Abandoned           Critical      1
1      Abandoned               High      1
4      Foundling               High      1
7      Neglected               High      1

Total active + closed residents: 60


In [4]:
# ── Social worker roster derivation ──────────────────────────────────────────
def derive_sw_roster(recordings_df, months_lookback=12):
    """Active SWs = those with >= 3 sessions at a safehouse in the last N months."""
    cutoff = SNAPSHOT - pd.DateOffset(months=months_lookback)
    recent = recordings_df[recordings_df['session_date'] >= cutoff].copy()
    # Join safehouse_id from residents
    recent = recent.merge(residents[['resident_id','safehouse_id']], on='resident_id', how='left')
    roster = (recent.groupby(['safehouse_id','social_worker'])
                    .size().reset_index(name='session_count'))
    return roster[roster['session_count'] >= 3]

sw_roster = derive_sw_roster(recordings)
print(f"Active social workers in roster: {len(sw_roster)}")
print(sw_roster.head(10).to_string())


Active social workers in roster: 161
   safehouse_id social_worker  session_count
0             1         SW-01             16
1             1         SW-02             15
2             1         SW-03             13
3             1         SW-04             11
4             1         SW-05             13
5             1         SW-06             10
6             1         SW-07             15
7             1         SW-08             14
8             1         SW-09             14
9             1         SW-10             11


In [5]:
# ── Service type distribution ─────────────────────────────────────────────────
print("Session type distribution:")
print(recordings['session_type'].value_counts())
print("\nTop interventions applied:")
print(recordings['interventions_applied'].value_counts().head(10))
print("\nProgress noted in sessions:")
print(recordings['progress_noted'].value_counts())


Session type distribution:
session_type
Individual    1805
Group         1014
Name: count, dtype: int64

Top interventions applied:
interventions_applied
Teaching                    246
Healing                     237
Legal Services              232
Caring                      218
Caring, Teaching             96
Legal Services, Caring       88
Healing, Teaching            88
Teaching, Healing            84
Teaching, Legal Services     80
Healing, Caring              80
Name: count, dtype: int64

Progress noted in sessions:
progress_noted
True     2638
False     181
Name: count, dtype: int64


## Phase 4 — Data Preparation: Composite Outcome Deltas (Ch. 2–3, 7)

In [6]:
# ── Compute 3-month composite outcome delta per recording ─────────────────────
WINDOW_DAYS = 90
W_HEALTH = 0.40; W_EDU = 0.35; W_INCIDENTS = 0.25

def compute_health_delta(resident_id, session_date, window_days=WINDOW_DAYS):
    h = health[health['resident_id']==resident_id].copy()
    h['record_date'] = pd.to_datetime(h['record_date'])
    baseline = h[h['record_date'] <= session_date]['general_health_score']
    followup = h[(h['record_date'] > session_date) &
                  (h['record_date'] <= session_date + pd.Timedelta(days=window_days))]['general_health_score']
    if baseline.empty or followup.empty:
        return None
    return float(followup.mean() - baseline.iloc[-1])

def compute_edu_delta(resident_id, session_date, window_days=WINDOW_DAYS):
    e = education[education['resident_id']==resident_id].copy()
    e['record_date'] = pd.to_datetime(e['record_date'])
    baseline = e[e['record_date'] <= session_date]['progress_percent']
    followup = e[(e['record_date'] > session_date) &
                  (e['record_date'] <= session_date + pd.Timedelta(days=window_days))]['progress_percent']
    if baseline.empty or followup.empty:
        return None
    return float(followup.mean() - baseline.iloc[-1])

def compute_incident_delta(resident_id, session_date, window_days=WINDOW_DAYS):
    inc = incidents[incidents['resident_id']==resident_id]
    before_count = len(inc[inc['incident_date'] <= session_date])
    after_count  = len(inc[(inc['incident_date'] > session_date) &
                           (inc['incident_date'] <= session_date + pd.Timedelta(days=window_days))])
    return float(after_count - before_count)   # negative = improvement

def composite_delta(dh, de, di):
    parts = [(dh, W_HEALTH), (de, W_EDU), ((-di if di is not None else None), W_INCIDENTS)]
    valid = [(v,w) for v,w in parts if v is not None]
    if not valid:
        return None
    total_w = sum(w for _,w in valid)
    return sum(v*w for v,w in valid) / total_w

print("Computing composite outcome deltas for each recording...")
print("(This may take a moment for larger datasets)")

session_rows = []
for idx, rec in recordings.iterrows():
    rid  = rec['resident_id']
    sdate = pd.to_datetime(rec['session_date'])

    dh = compute_health_delta(rid, sdate)
    de = compute_edu_delta(rid, sdate)
    di = compute_incident_delta(rid, sdate)
    comp = composite_delta(dh, de, di)

    session_rows.append({
        'recording_id':     rec['recording_id'],
        'resident_id':      rid,
        'session_date':     sdate,
        'social_worker':    rec['social_worker'],
        'session_type':     rec['session_type'],
        'interventions_applied': rec['interventions_applied'],
        'sessions_per_month': 1,   # will be aggregated per window
        'health_delta':     dh,
        'edu_delta':        de,
        'incident_delta':   di,
        'composite_delta':  comp,
    })

sessions_df = pd.DataFrame(session_rows)
sessions_df = sessions_df.merge(
    residents[['resident_id','safehouse_id','case_category','current_risk_level',
               'present_age','sub_cat_trafficked','sub_cat_physical_abuse',
               'sub_cat_sexual_abuse','sub_cat_osaec']],
    on='resident_id', how='left')

print(f"Total session records with outcomes: {sessions_df['composite_delta'].notna().sum()} / {len(sessions_df)}")
sessions_df[['composite_delta','health_delta','edu_delta','incident_delta']].describe().round(3)


Computing composite outcome deltas for each recording...
(This may take a moment for larger datasets)


Total session records with outcomes: 2819 / 2819


,composite_delta,health_delta,edu_delta,incident_delta
count,2819.0000,1083.0000,1083.0000,2819.0000
mean,1.8700,0.0610,8.6100,-0.7280
std,2.9490,0.0990,11.8660,1.3690
min,-4.5120,-0.2200,-12.2000,-5.0000
25%,0.0000,-0.0150,0.0000,-1.0000
50%,1.0000,0.0600,2.2000,0.0000
75%,2.9910,0.1230,17.3670,0.0000
max,15.1980,0.3530,42.8000,4.0000


In [7]:
# ── Build age bands ───────────────────────────────────────────────────────────
def build_age_band(age_str):
    try:
        years = int(str(age_str).split()[0])
        if years < 10:   return "Under 10"
        elif years <= 13: return "10-13"
        elif years <= 17: return "14-17"
        else:             return "18+"
    except:
        return "Unknown"

sessions_df['age_band'] = sessions_df['present_age'].apply(build_age_band)
sessions_df['case_category'] = sessions_df['case_category'].fillna('Unknown')
sessions_df['current_risk_level'] = sessions_df['current_risk_level'].fillna('Unknown')


## Phase 5 — Modelling: Rule-Based Clustering + Effectiveness Ranking (Ch. 12)

**Why rule-based clustering instead of K-Means?**

With ~60 active residents and 4 profile dimensions, K-Means produces unstable, non-interpretable clusters. A social worker cannot act on "Cluster 2" — they can act on "High-risk, Trafficked, Age 14–17." Rule-based hierarchical grouping is deterministic, human-interpretable, and collapsible gracefully when cells are too small (fewer than 5 members merge into the nearest parent group).

This is an example of choosing **interpretability appropriately** (Ch. 12): the clustering layer must be understandable to social workers; the ranking layer is data-driven.


In [8]:
# ── Profile label construction ────────────────────────────────────────────────
def build_abuse_flags(row):
    flags = []
    for col, label in [('sub_cat_trafficked','Trafficked'),
                       ('sub_cat_physical_abuse','Physical Abuse'),
                       ('sub_cat_sexual_abuse','Sexual Abuse'),
                       ('sub_cat_osaec','OSAEC')]:
        val = row.get(col, False)
        if val is True or str(val).lower() == 'true':
            flags.append(label)
    return ' + '.join(flags) if flags else 'No Sub-type'

def build_profile_label(row):
    return (f"{row['current_risk_level']} | {build_abuse_flags(row)} | "
            f"Age {row['age_band']}")

sessions_df['profile_label'] = sessions_df.apply(build_profile_label, axis=1)

# ── Hierarchical collapse for small clusters ───────────────────────────────────
cluster_counts = sessions_df['profile_label'].value_counts()

def assign_cluster(row, min_size=5):
    full = build_profile_label(row)
    if cluster_counts.get(full, 0) >= min_size:
        return full
    partial = f"{row['current_risk_level']} | {build_abuse_flags(row)}"
    if cluster_counts.get(partial, 0) >= min_size:
        return partial
    return row['current_risk_level']

sessions_df['profile_cluster'] = sessions_df.apply(assign_cluster, axis=1)
print("Profile clusters (after collapse):")
print(sessions_df['profile_cluster'].value_counts())


Profile clusters (after collapse):
profile_cluster
Low | No Sub-type | Age 10-13                                   344
Medium | No Sub-type | Age 14-17                                287
Low | No Sub-type | Age 18+                                     192
Medium | Trafficked | Age 14-17                                 127
High | Trafficked + Sexual Abuse | Age 14-17                    118
Low | Sexual Abuse | Age 10-13                                  114
High | No Sub-type | Age 14-17                                  106
Low | No Sub-type | Age 14-17                                    93
High | Sexual Abuse | Age 14-17                                  92
Low | OSAEC | Age 14-17                                          87
High | OSAEC | Age 14-17                                         86
Low | Sexual Abuse | Age 14-17                                   81
Medium | OSAEC | Age 18+                                         79
Medium | Sexual Abuse | Age 18+                                  

In [9]:
# ── Service outcomes by cluster ───────────────────────────────────────────────
service_cluster_outcomes = (
    sessions_df.dropna(subset=['composite_delta'])
    .groupby(['profile_cluster','interventions_applied'])['composite_delta']
    .agg(['mean','std','count'])
    .reset_index()
    .sort_values(['profile_cluster','mean'], ascending=[True, False])
)
print("Service effectiveness by cluster (top entries):")
print(service_cluster_outcomes.head(20).round(3).to_string())


Service effectiveness by cluster (top entries):
                       profile_cluster              interventions_applied    mean    std  count
19  Critical | No Sub-type | Age 14-17    Legal Services, Healing, Caring 10.5050    NaN      1
3   Critical | No Sub-type | Age 14-17             Caring, Legal Services  6.0560    NaN      1
12  Critical | No Sub-type | Age 14-17          Healing, Teaching, Caring  6.0560    NaN      1
18  Critical | No Sub-type | Age 14-17            Legal Services, Healing  6.0560    NaN      1
23  Critical | No Sub-type | Age 14-17  Legal Services, Teaching, Healing  5.1470 7.5760      2
6   Critical | No Sub-type | Age 14-17                   Caring, Teaching  4.1680 5.4880      3
31  Critical | No Sub-type | Age 14-17           Teaching, Legal Services  4.0380 3.4970      3
21  Critical | No Sub-type | Age 14-17           Legal Services, Teaching  3.8350 5.7980      3
0   Critical | No Sub-type | Age 14-17                             Caring  2.3520 3.2470

In [10]:
# ── Within-cluster ranking functions ─────────────────────────────────────────
def rank_services(cluster_df):
    r = (cluster_df.dropna(subset=['composite_delta'])
         .groupby('interventions_applied')['composite_delta']
         .mean().sort_values(ascending=False))
    return r.index.tolist()

def best_session_type(cluster_df):
    r = (cluster_df.dropna(subset=['composite_delta'])
         .groupby('session_type')['composite_delta'].mean())
    return r.idxmax() if not r.empty else None

def best_freq_band(cluster_df):
    c = cluster_df.copy()
    c['freq_band'] = pd.cut(c.get('sessions_per_month', pd.Series([3]*len(c))),
                             bins=[0,3,6,99], labels=['1-3','4-6','7+'])
    r = (c.dropna(subset=['composite_delta'])
          .groupby('freq_band', observed=True)['composite_delta'].mean())
    return str(r.idxmax()) if not r.empty else '4-6'

def best_sw(cluster_df, roster, safehouse_id):
    available = roster[roster['safehouse_id']==safehouse_id]['social_worker'].tolist()
    sw_perf = (cluster_df[cluster_df['social_worker'].isin(available)]
               .dropna(subset=['composite_delta'])
               .groupby('social_worker')['composite_delta']
               .agg(['mean','count']).reset_index())
    if sw_perf.empty:
        return None, None
    best = sw_perf.sort_values('mean', ascending=False).iloc[0]
    return best['social_worker'], round(float(best['mean']), 3)

# ── Build cluster recommendations ─────────────────────────────────────────────
cluster_recommendations = {}
for cluster_label, cdf in sessions_df.groupby('profile_cluster'):
    safehouse_id = cdf['safehouse_id'].mode()[0] if not cdf['safehouse_id'].empty else None
    sw, sw_score = best_sw(cdf, sw_roster, safehouse_id) if safehouse_id is not None else (None, None)
    cluster_recommendations[cluster_label] = {
        'recommended_services':    rank_services(cdf),
        'recommended_session_type':best_session_type(cdf),
        'sessions_per_month':      best_freq_band(cdf),
        'recommended_sw':          sw,
        'sw_outcome_score':        sw_score,
        'cluster_size':            len(cdf),
    }

print(f"\nCluster recommendations built for {len(cluster_recommendations)} clusters")
for cl, rec in list(cluster_recommendations.items())[:3]:
    print(f"\n  Cluster: {cl}")
    print(f"    Top services:  {rec['recommended_services'][:3]}")
    print(f"    Session type:  {rec['recommended_session_type']}")
    print(f"    Freq/month:    {rec['sessions_per_month']}")
    print(f"    Best SW:       {rec['recommended_sw']} (score={rec['sw_outcome_score']})")
    print(f"    Cluster size:  {rec['cluster_size']}")



Cluster recommendations built for 33 clusters

  Cluster: Critical | No Sub-type | Age 14-17
    Top services:  ['Legal Services, Healing, Caring', 'Legal Services, Healing', 'Caring, Legal Services']
    Session type:  Group
    Freq/month:    1-3
    Best SW:       SW-17 (score=5.49)
    Cluster size:  74

  Cluster: High | No Sub-type | Age 10-13
    Top services:  ['Caring, Healing', 'Legal Services, Teaching, Caring', 'Legal Services, Caring, Healing']
    Session type:  Group
    Freq/month:    1-3
    Best SW:       SW-03 (score=9.323)
    Cluster size:  42

  Cluster: High | No Sub-type | Age 14-17
    Top services:  ['Legal Services, Healing', 'Healing, Teaching, Legal Services', 'Healing, Caring, Legal Services']
    Session type:  Individual
    Freq/month:    1-3
    Best SW:       SW-02 (score=2.882)
    Cluster size:  106


## Phase 6 — Evaluation (Ch. 15)

In [11]:
# ── Cluster stability ─────────────────────────────────────────────────────────
cluster_sizes = sessions_df['profile_cluster'].value_counts()
print("Cluster size distribution:")
print(cluster_sizes)

def assign_confidence_tier(n):
    if n >= 15: return 'high'
    elif n >= 5: return 'medium'
    else: return 'low'

tier_dist = {k: assign_confidence_tier(v) for k,v in
             {cl: rec['cluster_size'] for cl, rec in cluster_recommendations.items()}.items()}
tier_counts = pd.Series(tier_dist).value_counts()
print("\nConfidence tier distribution:")
print(tier_counts)
coverage_medium_high = tier_counts.get('medium',0) + tier_counts.get('high',0)
total = len(cluster_recommendations)
print(f"\nMedium+high confidence coverage: {coverage_medium_high}/{total} = {coverage_medium_high/max(total,1):.1%}")


Cluster size distribution:
profile_cluster
Low | No Sub-type | Age 10-13                                   344
Medium | No Sub-type | Age 14-17                                287
Low | No Sub-type | Age 18+                                     192
Medium | Trafficked | Age 14-17                                 127
High | Trafficked + Sexual Abuse | Age 14-17                    118
Low | Sexual Abuse | Age 10-13                                  114
High | No Sub-type | Age 14-17                                  106
Low | No Sub-type | Age 14-17                                    93
High | Sexual Abuse | Age 14-17                                  92
Low | OSAEC | Age 14-17                                          87
High | OSAEC | Age 14-17                                         86
Low | Sexual Abuse | Age 14-17                                   81
Medium | OSAEC | Age 18+                                         79
Medium | Sexual Abuse | Age 18+                                  79
Low |

In [12]:
# ── Recommendation lift: does top service outperform others? ─────────────────
lift_rows = []
for cluster, rec in cluster_recommendations.items():
    cdf = sessions_df[sessions_df['profile_cluster']==cluster].dropna(subset=['composite_delta'])
    top_svc = rec['recommended_services'][0] if rec['recommended_services'] else None
    if top_svc is None or len(cdf) < 3:
        continue
    recommended_mean = cdf[cdf['interventions_applied']==top_svc]['composite_delta'].mean()
    others_mean      = cdf[cdf['interventions_applied']!=top_svc]['composite_delta'].mean()
    if pd.isna(recommended_mean) or pd.isna(others_mean) or abs(others_mean) < 0.001:
        continue
    lift = (recommended_mean - others_mean) / max(abs(others_mean), 0.001) * 100
    lift_rows.append({'cluster':cluster,'top_service':top_svc,
                      'recommended_mean':round(recommended_mean,4),
                      'others_mean':round(others_mean,4),'lift_pct':round(lift,1)})

if lift_rows:
    lift_df = pd.DataFrame(lift_rows)
    print("Recommendation lift analysis:")
    print(lift_df.to_string())
    print(f"\nMean lift of recommended service vs others: {lift_df['lift_pct'].mean():.1f}%")
else:
    print("Lift analysis requires multiple service types per cluster.")
    print("With the current dataset size, most clusters have limited service variety.")


Recommendation lift analysis:
                                                         cluster                        top_service  recommended_mean  others_mean  lift_pct
0                             Critical | No Sub-type | Age 14-17    Legal Services, Healing, Caring           10.5047       1.6456  538.3000
1                                 High | No Sub-type | Age 10-13                    Caring, Healing           10.6310       4.8935  117.2000
2                                 High | No Sub-type | Age 14-17            Legal Services, Healing            3.1480       1.7020   85.0000
3                                       High | OSAEC | Age 14-17                   Caring, Teaching           15.1980       4.8386  214.1000
4                                High | Sexual Abuse | Age 14-17    Legal Services, Caring, Healing            5.5083       2.5442  116.5000
5                   High | Trafficked + Sexual Abuse | Age 14-17                            Healing            2.9399       

## Phase 7 — Interpretation: top_outcome_factors (Ch. 16)

In [13]:
# ── Build human-readable top_outcome_factors ─────────────────────────────────
def build_top_outcome_factors(cluster_label, services, session_type, freq, sw, sw_score, size):
    factors = []
    if services:
        factors.append(f"{services[0]} sessions show strongest composite improvements "
                       f"for the '{cluster_label}' profile")
    if session_type and freq:
        factors.append(f"{session_type} sessions at ~{freq}/month outperform other formats")
    if sw and sw_score is not None:
        factors.append(f"{sw} has the highest average outcome improvement for similar profiles "
                       f"(score: {sw_score:.2f})")
    if not factors:
        factors.append(f"Based on {size} similar residents in the '{cluster_label}' cluster")
    return factors[:3]

# ── Score all active residents ─────────────────────────────────────────────────
active = residents[residents['case_status'].isin(['Active','active'])].copy()
active['age_band'] = active['present_age'].apply(build_age_band)

scored_residents = []
for _, res in active.iterrows():
    cluster = assign_cluster(res)
    rec = cluster_recommendations.get(cluster, {})
    safehouse_id = res['safehouse_id']
    sw, sw_score = (rec.get('recommended_sw'), rec.get('sw_outcome_score'))
    size = rec.get('cluster_size', 0)
    factors = build_top_outcome_factors(
        cluster,
        rec.get('recommended_services', []),
        rec.get('recommended_session_type'),
        rec.get('sessions_per_month'),
        sw, sw_score, size
    )
    freq_label = rec.get('sessions_per_month', '4-6')
    sessions_int = {'1-3':2,'4-6':5,'7+':8}.get(str(freq_label), 5)

    scored_residents.append({
        'resident_id':                   int(res['resident_id']),
        'profile_cluster':               cluster,
        'recommended_services':          rec.get('recommended_services', [])[:3],
        'recommended_session_type':      rec.get('recommended_session_type'),
        'recommended_sessions_per_month':sessions_int,
        'recommended_social_worker':     sw,
        'sw_outcome_score':              sw_score,
        'similar_resident_count':        size,
        'confidence_tier':               assign_confidence_tier(size),
        'top_outcome_factors':           factors,
        'scored_at':                     datetime.now().isoformat(),
    })

recs_df = pd.DataFrame(scored_residents)
print(f"Scored {len(recs_df)} active residents")
print("\nConfidence tier distribution:")
print(recs_df['confidence_tier'].value_counts())
print("\nSample recommendations:")
print(recs_df[['resident_id','profile_cluster','recommended_services',
               'confidence_tier','top_outcome_factors']].head(5).to_string())


Scored 30 active residents

Confidence tier distribution:
confidence_tier
high    30
Name: count, dtype: int64

Sample recommendations:
   resident_id                  profile_cluster                                                                             recommended_services confidence_tier                                                                                                                                                                                                                                                              top_outcome_factors
0            1   High | No Sub-type | Age 14-17    [Legal Services, Healing, Healing, Teaching, Legal Services, Healing, Caring, Legal Services]            high      [Legal Services, Healing sessions show strongest composite improvements for the 'High | No Sub-type | Age 14-17' profile, Individual sessions at ~1-3/month outperform other formats, SW-02 has the highest average outcome improvement for similar profiles (score: 2.8

## Phase 8 — Deployment (Ch. 17)

In [14]:
# ── Serialize model artifact ──────────────────────────────────────────────────
model_version = datetime.now().strftime('%Y%m%d_%H%M')
artifact = {
    'model_version':          model_version,
    'cluster_recommendations': cluster_recommendations,
    'sw_roster':              sw_roster.to_dict(orient='records'),
}
with open('/tmp/interventions_model.pkl','wb') as f:
    pickle.dump(artifact, f)

print("Intervention model artifact saved.")
print(f"  Clusters: {len(cluster_recommendations)}")
print(f"  Residents scored: {len(recs_df)}")
print(f"  Model version: {model_version}")

# Production gate: coverage check
cov_rate = (recs_df['confidence_tier'].isin(['medium','high'])).mean()
print(f"\nMedium+high confidence coverage: {cov_rate:.1%} (target: ≥70%)")

deployment_note = """
DEPLOYMENT PATTERN (production):

  Supabase: residents + process_recordings + health + education + incidents
      ↓
  train_interventions.py  (Azure Container Apps Job — 1st of each month at 02:00 UTC)
      ↓  saves cluster_recommendations + sw_roster
  Azure Blob Storage: interventions_model.pkl
      ↓  loaded by
  score_interventions.py  (triggered via POST /score/interventions)
      ↓  upserts per-resident recommendations
  Supabase: intervention_recommendations (RLS: Admin + SocialWorker only)
      ↓
  .NET API → social worker dashboard

IMPORTANT LIMITATIONS:
  - With ~60 active residents, most clusters will be low/medium confidence.
    Recommendations improve meaningfully when population exceeds ~100 active cases.
  - All recommendations are decision-support only — no automated actions.
  - Outcome improvements are correlational, not causal.
"""
print(deployment_note)


Intervention model artifact saved.
  Clusters: 33
  Residents scored: 30
  Model version: 20260409_1802

Medium+high confidence coverage: 100.0% (target: ≥70%)

DEPLOYMENT PATTERN (production):

  Supabase: residents + process_recordings + health + education + incidents
      ↓
  train_interventions.py  (Azure Container Apps Job — 1st of each month at 02:00 UTC)
      ↓  saves cluster_recommendations + sw_roster
  Azure Blob Storage: interventions_model.pkl
      ↓  loaded by
  score_interventions.py  (triggered via POST /score/interventions)
      ↓  upserts per-resident recommendations
  Supabase: intervention_recommendations (RLS: Admin + SocialWorker only)
      ↓
  .NET API → social worker dashboard

IMPORTANT LIMITATIONS:
  - With ~60 active residents, most clusters will be low/medium confidence.
    Recommendations improve meaningfully when population exceeds ~100 active cases.
  - All recommendations are decision-support only — no automated actions.
  - Outcome improvements are